# Example Usage of Marlowe Oracle

This example uses the Marlowe oracle to read a random value into a contract and pay that amount of ada to a recipient.

The oracle resides at `addr_test1qzy4aluz9zcmxuhuqgjhl457ee0ma0hcq38w62ckaagjwgvnjlzdfawd9kvva673fht7737e3r5j322v7090uqhpn0wq7a2gly` on the `preprod` network. See https://github.com/input-output-hk/marlowe-cardano/blob/main/marlowe-apps/Oracle.md for instructions on using the oracle.

![Oracle Example](example-oracle-contract.png)

## Setup

### Select the preproduction network

In [ ]:
export CARDANO_NODE_SOCKET_PATH=node.socket
export CARDANO_TESTNET_MAGIC=1

### Set the location of Marlowe Runtime

In [ ]:
export MARLOWE_RT_HOST=127.0.0.1
export MARLOWE_RT_PORT=13700

### Set the location of Marlowe executables

In [ ]:
MARLOWE_RT_CLI=marlowe-runtime-cli
MARLOWE_CLI=marlowe-cli

### Constants

In [ ]:
ADA=1000000

## Addresses

### Oracle address

In [ ]:
ORACLE_ADDR=addr_test1qzy4aluz9zcmxuhuqgjhl457ee0ma0hcq38w62ckaagjwgvnjlzdfawd9kvva673fht7737e3r5j322v7090uqhpn0wq7a2gly

### Address for "c.marlowe" role

In [ ]:
PAYMENT_ADDR=$(cat payment.testnet.address)
PAYMENT_SKEY=payment.skey
echo "PAYMENT_ADDR = $PAYMENT_ADDR"

In [ ]:
ROLE_POLICY=8bb3b343d8e404472337966a722150048c768d0a92a9813596c5338d
ROLE_NAME=c.marlowe

### Address for recipient of funds

In [ ]:
RECIPIENT_ADDR=addr_test1qq9prvx8ufwutkwxx9cmmuuajaqmjqwujqlp9d8pvg6gupcvluken35ncjnu0puetf5jvttedkze02d5kf890kquh60slacjyp

## Contract

In [ ]:
yaml2json << EOI > example-oracle-contract.json
timeout: $((1000 * (`date -u +%s` + 1 * 60 * 60)))
timeout_continuation: close
when:
- case:
    deposits:
      multiply: 100
      times: 1000000
    into_account:
      role_token: $ROLE_NAME
    of_token:
      currency_symbol: ''
      token_name: ''
    party:
      role_token: c.marlowe
  then:
    timeout: $((1000 * (`date -u +%s` + 2 * 60 * 60)))
    timeout_continuation: close
    when:
    - case:
        choose_between:
        - from: 50
          to: 100
        for_choice:
          choice_name: RANDOM
          choice_owner:
            address: "$ORACLE_ADDR"
      then:
        from_account:
          role_token: c.marlowe
        pay:
          multiply:
            value_of_choice:
              choice_name: RANDOM
              choice_owner:
                address: "$ORACLE_ADDR"
          times: 1000000
        then: close
        to:
          party:
            address: "$RECIPIENT_ADDR"
        token:
          currency_symbol: ''
          token_name: ''
EOI
cat example-oracle-contract.json

## Transactions

### 1. Create the contract

In [ ]:
CONTRACT_ID=$(
"$MARLOWE_RT_CLI" create \
  --core-file example-oracle-contract.json \
  --role-token-policy-id "$ROLE_POLICY" \
  --min-utxo "$((2 * ADA))" \
  --change-address "$PAYMENT_ADDR" \
  --manual-sign tx-1.unsigned \
  2> /dev/null \
| jq -r 'fromjson | .contractId' \
)
echo "CONTRACT_ID = $CONTRACT_ID"

In [ ]:
"$MARLOWE_CLI" transaction submit \
  --tx-body-file tx-1.unsigned \
  --required-signer "$PAYMENT_SKEY" \
  --timeout 600s

### 2. Deposit the 100 ada

In [ ]:
"$MARLOWE_RT_CLI" deposit \
  --contract "$CONTRACT_ID" \
  --from-party "$ROLE_NAME" \
  --to-party "$ROLE_NAME" \
  --lovelace "$((100 * ADA))" \
  --change-address "$PAYMENT_ADDR" \
  --manual-sign tx-2.unsigned \
| jq 'fromjson' | json2yaml

In [ ]:
$MARLOWE_CLI transaction submit \
  --tx-body-file tx-2.unsigned \
  --required-signer "$PAYMENT_SKEY" \
  --timeout 600s

### Wait for the oracle to report a random value to the contract

In [ ]:
sleep 3m

### 3. View the report by the oracle

In [ ]:
"$MARLOWE_RT_CLI" log \
  --show-contract \
  "$CONTRACT_ID"

We see that the oracle has reported the value 91.